# Chapter 15 – Worked examples

This notebook runs the examples of [Chapter 15](../index.md) live. The code lives in the modules next to it
(`ac_jacobian.py`, `nr_power_flow.py`, `jacobian_timing.py`, `powsybl_slack.py`); the cells only call them, so a
change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository. The Octave example (`matpower_pf.m`) runs in Octave, not here.

In [ ]:
import ac_jacobian
import jacobian_timing
import nr_power_flow
import numpy as np
import powsybl_slack

## Injections and Jacobian blocks in matrix form

A line of $0.01 + j0.1$ pu between a source at $1\angle 0$ and a load at $0.98\angle(-0.05\ \text{rad})$. The
injections are $S = [V]\,\overline{Y V}$; the first block, $J_{11} = \partial P / \partial\theta$, is the real part
of $j[V]\,\overline{([I] - Y[V])}$.

In [ ]:
y_line = 1 / (0.01 + 0.1j)
y = np.array([[y_line, -y_line], [-y_line, y_line]])
theta, vm = np.array([0.0, -0.05]), np.array([1.0, 0.98])
print('S =', np.round(ac_jacobian.injections(theta, vm, y), 4))
j11, j12, j21, j22 = ac_jacobian.jacobian_blocks(theta, vm, y)
print('J11 =', np.round(j11, 4).tolist())
print('J22 =', np.round(j22, 4).tolist())

## Check the Jacobian against finite differences

Never trust a Jacobian you have not differenced. `finite_difference_error` compares every column of the four
blocks with central differences of `injections`, here on the converged state of IEEE 14 (Ybus from pandapower).

In [ ]:
th14, vm14, y14 = jacobian_timing.case_state('case14')
err = jacobian_timing.finite_difference_error(th14, vm14, y14.toarray())
print(f'IEEE 14: max |analytic - central difference| = {err:.1e}')

## Newton–Raphson on the five-bus grid

`nr_power_flow.solve` assembles the reduced Jacobian from the four blocks (rows of $P$ for the non-slack buses,
rows of $Q$ for the PQ buses) and iterates. The mismatch norm shows the quadratic convergence of Chapter 6.

In [ ]:
theta5, vm5, history, ok = nr_power_flow.solve(nr_power_flow.CASE5)
print('mismatch inf-norm per iteration:', ' '.join(f'{h:.1e}' for h in history), '| converged:', ok)
print('angles (deg):', np.round(np.rad2deg(theta5), 3))
print('magnitudes  :', np.round(vm5, 4))

## Distributed slack

With the slack distributed, the losses are shared by the participating generators with weights $k_i$,
$\sum k_i = 1$. OpenLoadFlow's balance type selects the weights; on IEEE 14 with the loads raised by 10 %,
`proportional to P` shares in the ratio of the targets (232.4 : 40) and `proportional to Pmax` equally (both
generating units have `max_p = 9999` MW). With a single slack the mismatch stays on the slack bus.

In [ ]:
gens, flows = powsybl_slack.compare(1.1)
print(gens.round(2))
print(flows.round(2))